In [1]:
#%env XLA_PYTHON_CLIENT_MEM_FRACTION=.25
%env JAX_LOG_COMPILES=1

env: JAX_LOG_COMPILES=1


In [2]:
import jax
jax.config.update('jax_threefry_partitionable', True)
import netket as nk

import netket.experimental
from functools import partial

# from jax.config import config
# config.update("jax_enable_x64", False)
# del config

Finished tracing + transforming jit(convert_element_type) in 0.0002925395965576172 sec
Finished tracing + transforming jit(broadcast_in_dim) in 0.0002760887145996094 sec
Compiling broadcast_in_dim for with global shapes and types [ShapedArray(float64[])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(broadcast_in_dim) in 0.0014832019805908203 sec
Finished XLA compilation of jit(broadcast_in_dim) in 1.9011991024017334 sec
Finished tracing + transforming jit(broadcast_in_dim) in 0.00022029876708984375 sec
Compiling broadcast_in_dim for with global shapes and types [ShapedArray(float64[])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(broadcast_in_dim) in 0.0009856224060058594 sec
Finished XLA compilation of jit(broadcast_in_dim) in 0.010766029357910156 sec
Finished tracing + transforming jit(convert_element_type) in 0.00017952919006347656 sec
Finished tracing + transforming jit(convert_elem

In [3]:
jax.devices()

[gpu(id=0), gpu(id=1)]

In [4]:
L = 32

n_chains_per_device = 512

n_discard = 0 # to be fair comparison we set discard to 0, as it's per chain
# TODO later increase Ns and chains beyond what cuda can handle in paralell, and turn back on discard

g = nk.graph.Hypercube(length=L, n_dim=1, pbc=True)
hi = nk.hilbert.Spin(s=1 / 2, N=g.n_nodes)
ha = nk.operator.Ising(hilbert=hi, graph=g, h=1.0)
ma = nk.models.RBM(alpha=8, param_dtype=complex)
#ma = nk.models.GCNN(g, features=8, layers=4, param_dtype=complex, mode='fft')
sa1 = nk.sampler.MetropolisLocal(hi, n_chains=n_chains_per_device)
sa2 = nk.sampler.MetropolisLocal(hi, n_chains=n_chains_per_device*jax.local_device_count())

op = nk.optimizer.Sgd(learning_rate=0.1)

In [5]:
sr = nk.optimizer.SR(diag_shift=0.01, qgt=nk.optimizer.qgt.QGTOnTheFly)
srp = nk.optimizer.SR(diag_shift=0.01, qgt=partial(nk.optimizer.qgt.QGTJacobianPyTree, holomorphic=True))

we create 2 vstates:
- vs1 using 1 gpu
- vs2 using 2 local gpus

note that so far this most likely only works with local devices, as we don't sync the PRNG for the non pjit stuff yet

In [6]:
vs1 = nk.vqs.MCState(sa1, ma, n_samples=8192, n_discard_per_chain=n_discard)
sampler_state1 =  vs1.sampler_state

sharding = jax.sharding.PositionalSharding(jax.devices())
vs2 = nk.vqs.MCState(sa2, ma, n_samples=8192, n_discard_per_chain=n_discard)
sampler_state2 = vs2.sampler_state.replace(σ=jax.device_put(vs2.sampler_state.σ, sharding.reshape(-1, 1)))
vs2.sampler_state = sampler_state2
# 
# x = vs.samples
# p = vs.variables
# f = vs._apply_fun
# lowered = jax.jit(jax.vmap(f, in_axes=(None, 0))).lower(p, x)
# compiled = lowered.compile()
# ca = compiled.cost_analysis()
# intensity = ca[0]['flops'] / ca[0]['bytes accessed']
# print('Comp. intensity fwd pass:', intensity, 'flops/byte')
#

Finished tracing + transforming jit(convert_element_type) in 0.0001766681671142578 sec
Finished tracing + transforming <lambda> for pjit in 0.0003371238708496094 sec
Finished tracing + transforming _threefry_seed for pjit in 0.0018298625946044922 sec
Compiling _threefry_seed for with global shapes and types [ShapedArray(int64[])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(_threefry_seed) in 0.0020782947540283203 sec
Finished XLA compilation of jit(_threefry_seed) in 0.045128822326660156 sec
Finished tracing + transforming jit(broadcast_in_dim) in 0.00021409988403320312 sec
Compiling broadcast_in_dim for with global shapes and types [ShapedArray(float64[])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(broadcast_in_dim) in 0.0010089874267578125 sec
Finished XLA compilation of jit(broadcast_in_dim) in 0.041947126388549805 sec
Finished tracing + transforming <lambda> for pjit in 0.000288

Finished XLA compilation of jit(dynamic_slice) in 0.01112818717956543 sec
Finished tracing + transforming jit(squeeze) in 0.00019478797912597656 sec
Compiling squeeze for with global shapes and types [ShapedArray(uint32[1,2])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(squeeze) in 0.0011131763458251953 sec
Finished XLA compilation of jit(squeeze) in 0.010786771774291992 sec
Compiling _threefry_split_foldlike for with global shapes and types [ShapedArray(uint32[2])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(_threefry_split_foldlike) in 0.0024297237396240234 sec
Finished XLA compilation of jit(_threefry_split_foldlike) in 0.05825662612915039 sec
Finished tracing + transforming _unstack for pjit in 0.0005328655242919922 sec
Compiling _unstack for with global shapes and types [ShapedArray(uint32[2,2])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module co

### benchmark the sampler

In [7]:
from netket.sampler.metropolis import MetropolisLocal
from functools import partial

In [8]:
@partial(jax.jit, static_argnums=(1, 4))
def _sample_chain(sampler, machine, parameters, state, chain_length):
    state, samples = jax.lax.scan(
        lambda state, _: sampler.sample_next(machine, parameters, state),
        state,
        xs=None,
        length=chain_length,
    )

    return samples, state

In [9]:
x1 = jax.block_until_ready(_sample_chain(sa1, ma, vs1.variables, sampler_state1, 128))

Finished tracing + transforming fn for pjit in 0.00028252601623535156 sec
Finished tracing + transforming real for pjit in 0.00018525123596191406 sec
Finished tracing + transforming right_shift for pjit in 0.0003781318664550781 sec
Finished tracing + transforming signbit for pjit in 0.0013871192932128906 sec
Finished tracing + transforming fn for pjit in 0.0003371238708496094 sec
Finished tracing + transforming fn for pjit in 0.0002765655517578125 sec
Finished tracing + transforming fn for pjit in 0.0003151893615722656 sec
Finished tracing + transforming fn for pjit in 0.00031566619873046875 sec
Finished tracing + transforming <lambda> for pjit in 0.00020623207092285156 sec
Finished tracing + transforming <lambda> for pjit in 0.00020599365234375 sec
Finished tracing + transforming fn for pjit in 0.0002639293670654297 sec
Finished tracing + transforming <lambda> for pjit in 0.00037932395935058594 sec
Finished tracing + transforming _reduce_sum for pjit in 0.0003535747528076172 sec
Finis

In [10]:
x2 = jax.block_until_ready(_sample_chain(sa2, ma, vs2.variables, sampler_state2, 128/jax.local_device_count()))

Finished tracing + transforming fn for pjit in 0.0002696514129638672 sec
Finished tracing + transforming real for pjit in 0.00018072128295898438 sec
Finished tracing + transforming right_shift for pjit in 0.000308990478515625 sec
Finished tracing + transforming signbit for pjit in 0.0012881755828857422 sec
Finished tracing + transforming fn for pjit in 0.00037550926208496094 sec
Finished tracing + transforming fn for pjit in 0.00026154518127441406 sec
Finished tracing + transforming fn for pjit in 0.0003082752227783203 sec
Finished tracing + transforming fn for pjit in 0.00031065940856933594 sec
Finished tracing + transforming <lambda> for pjit in 0.00019860267639160156 sec
Finished tracing + transforming <lambda> for pjit in 0.00019693374633789062 sec
Finished tracing + transforming fn for pjit in 0.0002560615539550781 sec
Finished tracing + transforming <lambda> for pjit in 0.0003001689910888672 sec
Finished tracing + transforming _reduce_sum for pjit in 0.00040268898010253906 sec
Fi

In [11]:
%timeit _ = jax.block_until_ready(_sample_chain(sa1, ma, vs1.variables, sampler_state1, 128))

888 ms ± 3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%timeit _ = jax.block_until_ready(_sample_chain(sa2, ma, vs2.variables, sampler_state2, 128/jax.local_device_count()))

479 ms ± 84.8 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [14]:
sap = netket.experimental.sampler.MetropolisSamplerPmap(hi, nk.sampler.rules.LocalRule(), n_chains=n_chains_per_device*jax.local_device_count())

In [15]:
sampler_statep = sap.init_state(ma, vs1.variables)

Finished tracing + transforming <lambda> for pjit in 0.0002646446228027344 sec
Finished tracing + transforming <lambda> for pjit in 0.00025177001953125 sec
Finished tracing + transforming clip for pjit in 0.0021445751190185547 sec
Finished tracing + transforming <lambda> for pjit in 0.0002510547637939453 sec
Finished tracing + transforming <lambda> for pjit in 0.00024819374084472656 sec
Finished tracing + transforming clip for pjit in 0.0017888545989990234 sec
Finished tracing + transforming <lambda> for pjit in 0.0003211498260498047 sec
Finished tracing + transforming fn for pjit in 0.00025582313537597656 sec
Finished tracing + transforming fn for pjit in 0.00025200843811035156 sec
Finished tracing + transforming <lambda> for pjit in 0.00024819374084472656 sec
Finished tracing + transforming _randint for pjit in 0.010978937149047852 sec
Finished tracing + transforming fn for pjit in 0.000301361083984375 sec
Finished tracing + transforming <lambda> for pjit in 0.0002982616424560547 sec

In [16]:
xp = jax.block_until_ready(_sample_chain(sap, ma, vs1.variables, sampler_statep, 128/jax.local_device_count()))

Finished tracing + transforming <lambda> for pjit in 0.0002884864807128906 sec
Finished tracing + transforming <lambda> for pjit in 0.0002701282501220703 sec
Finished tracing + transforming fn for pjit in 0.0002868175506591797 sec
Finished tracing + transforming fn for pjit in 0.0002815723419189453 sec
Finished tracing + transforming _uniform for pjit in 0.004775524139404297 sec
Finished tracing + transforming _normal_real for pjit in 0.0057299137115478516 sec
Finished tracing + transforming fn for pjit in 0.0003097057342529297 sec
Finished tracing + transforming fn for pjit in 0.0002703666687011719 sec
Finished tracing + transforming true_divide for pjit in 0.00032639503479003906 sec
Finished tracing + transforming _normal for pjit in 0.00984048843383789 sec
Finished tracing + transforming fn for pjit in 0.0003094673156738281 sec
Finished tracing + transforming <lambda> for pjit in 0.00026535987854003906 sec
Finished tracing + transforming <lambda> for pjit in 0.0003528594970703125 se

In [17]:
%timeit _ = jax.block_until_ready(_sample_chain(sap, ma, vs1.variables, sampler_statep, 128/jax.local_device_count()))

487 ms ± 81.2 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [18]:
x1[0].shape, x2[0].shape, xp[0].shape

((128, 512, 32), (64, 1024, 32), (64, 1024, 32))

In [19]:
x1[0].sharding, x2[0].sharding, xp[0].sharding

(SingleDeviceSharding(device=gpu(id=0)),
 PositionalSharding([[[{GPU 0}]
                      [{GPU 1}]]]),
 GSPMDSharding({replicated}))

In [20]:
x2[0].sharding.shape

(1, 2, 1)

In [21]:
x1 = jax.block_until_ready(vs1.sample())
x2 = jax.block_until_ready(vs2.sample())
x1 = jax.block_until_ready(vs1.sample())
x2 = jax.block_until_ready(vs2.sample())

Finished tracing + transforming _sample_chain for pjit in 0.027802467346191406 sec
Compiling _sample_chain for with global shapes and types [ShapedArray(int64[], weak_type=True), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32]), ShapedArray(float64[512,32]), ShapedArray(uint32[2]), ShapedArray(int64[], weak_type=True), ShapedArray(int64[], weak_type=True)]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(_sample_chain) in 0.0522465705871582 sec
Finished XLA compilation of jit(_sample_chain) in 0.459089994430542 sec
Finished tracing + transforming _sample_chain for pjit in 0.027747392654418945 sec
Compiling _sample_chain for with global shapes and types [ShapedArray(int64[], weak_type=True), ShapedArr

In [22]:
%timeit jax.block_until_ready(vs1.sample())

112 ms ± 4.04 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [23]:
%timeit jax.block_until_ready(vs2.sample())

61.2 ms ± 6 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### benchmark gradients (includes numba operator on the cpu)

In [24]:
eg1 = jax.block_until_ready(vs1.expect_and_grad(ha))
eg2 = jax.block_until_ready(vs2.expect_and_grad(ha))

Finished tracing + transforming jit(reshape) in 0.00020647048950195312 sec
Compiling reshape for with global shapes and types [ShapedArray(float64[16,512,32])]. Argument mapping: (GSPMDSharding({replicated}),).
Finished jaxpr to MLIR module conversion jit(reshape) in 0.0010373592376708984 sec
Finished XLA compilation of jit(reshape) in 0.01162409782409668 sec
Finished tracing + transforming atleast_2d for pjit in 0.00020933151245117188 sec
Finished tracing + transforming fn for pjit in 0.0002753734588623047 sec
Finished tracing + transforming real for pjit in 0.00017905235290527344 sec
Finished tracing + transforming right_shift for pjit in 0.0003085136413574219 sec
Finished tracing + transforming signbit for pjit in 0.0013489723205566406 sec
Finished tracing + transforming fn for pjit in 0.00030875205993652344 sec
Finished tracing + transforming fn for pjit in 0.00026035308837890625 sec
Finished tracing + transforming fn for pjit in 0.0003025531768798828 sec
Finished tracing + transfo

Finished tracing + transforming _reduce_sum for pjit in 0.0003120899200439453 sec
Finished tracing + transforming _mean for pjit in 0.001199483871459961 sec
Finished tracing + transforming <lambda> for pjit in 0.0002543926239013672 sec
Finished tracing + transforming absolute for pjit in 0.07888555526733398 sec
Finished tracing + transforming _power for pjit in 0.0003380775451660156 sec
Finished tracing + transforming _reduce_sum for pjit in 0.00033783912658691406 sec
Finished tracing + transforming _statistics for pjit in 0.10714387893676758 sec
Finished tracing + transforming <lambda> for pjit in 0.0002551078796386719 sec
Finished tracing + transforming atleast_2d for pjit in 0.00016164779663085938 sec
Finished tracing + transforming conjugate for pjit in 0.00018262863159179688 sec
Finished tracing + transforming true_divide for pjit in 0.0003027915954589844 sec
Finished tracing + transforming forces_expect_hermitian for pjit in 0.20305418968200684 sec
Compiling forces_expect_hermiti

In [25]:
%timeit eg1 = jax.block_until_ready(vs1.expect_and_grad(ha))

104 ms ± 271 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [26]:
%timeit eg2 = jax.block_until_ready(vs2.expect_and_grad(ha))

70.1 ms ± 36.5 µs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### benchmark SR

In [27]:
S1 = vs1.quantum_geometric_tensor(sr.qgt_constructor)
S1p = vs1.quantum_geometric_tensor(srp.qgt_constructor)
S2 = vs2.quantum_geometric_tensor(sr.qgt_constructor)
S2p = vs2.quantum_geometric_tensor(srp.qgt_constructor)

Finished tracing + transforming mat_vec_factory for pjit in 0.011858463287353516 sec
Compiling mat_vec_factory for with global shapes and types [ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(float64[16,512,32])]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(mat_vec_factory) in 0.004536867141723633 sec
Finished XLA compilation of jit(mat_vec_factory) in 0.1364760398864746 sec
Finished tracing + transforming jacobian_default_mode for pjit in 0.0002853870391845703 sec
Finished tracing + transforming _reduce_sum for pjit in 0.0004112720489501953 sec
Finished tracing + transforming _mean for pjit in 0.0015184879302978516 sec
Finished tracing + transforming true_divide for pjit in 0.0003123283386230469 sec
Finished tracing + transforming <lambda> for pjit in 0.0002627372741699219 sec
Finished tracing + transforming true_divide for pjit in 0.00029468536376953

In [28]:
jax.tree_util.tree_leaves(S1p.O)[1].sharding

SingleDeviceSharding(device=gpu(id=0))

In [29]:
jax.tree_util.tree_leaves(S2p.O)[1].sharding.shape

(1, 2, 1, 1)

In [30]:
jax.tree_util.tree_leaves(S1._mat_vec)[0].sharding

SingleDeviceSharding(device=gpu(id=0))

In [31]:
jax.tree_util.tree_leaves(S2._mat_vec)[0].sharding.shape

(1, 2, 1)

#### otf

In [32]:
@jax.jit
def mv(S, v):
    return S@v

In [33]:
_ = jax.block_until_ready(mv(S1, vs1.parameters))
_ = jax.block_until_ready(mv(S1p, vs1.parameters))

Finished tracing + transforming fn for pjit in 0.0003762245178222656 sec
Finished tracing + transforming _reduce_sum for pjit in 0.0004096031188964844 sec
Finished tracing + transforming _mean for pjit in 0.0014240741729736328 sec
Finished tracing + transforming true_divide for pjit in 0.0002951622009277344 sec
Finished tracing + transforming <lambda> for pjit in 0.0002541542053222656 sec
Finished tracing + transforming fn for pjit in 0.00029659271240234375 sec
Finished tracing + transforming fn for pjit in 0.0003502368927001953 sec
Finished tracing + transforming fn for pjit in 0.0002951622009277344 sec
Finished tracing + transforming onthefly_mat_treevec for pjit in 0.021672964096069336 sec
Finished tracing + transforming mv for pjit in 0.022907257080078125 sec
Compiling mv for with global shapes and types [ShapedArray(float64[], weak_type=True), ShapedArray(complex128[16,512,32]), ShapedArray(complex128[16,512,256]), ShapedArray(complex128[]), ShapedArray(complex128[16,512,256]), Sh

In [34]:
%timeit jax.block_until_ready(mv(S1, vs1.parameters))

4.68 ms ± 11.8 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [35]:
%timeit jax.block_until_ready(mv(S2, vs2.parameters))

Finished tracing + transforming fn for pjit in 0.0003268718719482422 sec
Finished tracing + transforming _reduce_sum for pjit in 0.00040841102600097656 sec
Finished tracing + transforming _mean for pjit in 0.0014634132385253906 sec
Finished tracing + transforming <lambda> for pjit in 0.0002696514129638672 sec
Finished tracing + transforming onthefly_mat_treevec for pjit in 0.018632888793945312 sec
Finished tracing + transforming mv for pjit in 0.01985931396484375 sec
Compiling mv for with global shapes and types [ShapedArray(float64[], weak_type=True), ShapedArray(complex128[8,1024,32]), ShapedArray(complex128[8,1024,256]), ShapedArray(complex128[]), ShapedArray(complex128[8,1024,256]), ShapedArray(complex128[8,1024,256]), ShapedArray(complex128[8,1024,32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32])]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({devices=[1,2,1]0,1}), GSPMDSharding({devices=[1,2,1]0,1}), GSPMDSharding({r

8.77 ms ± 119 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [36]:
jvp_fn, = S1._mat_vec.args

In [37]:
from netket.optimizer.qgt.qgt_onthefly_logic import *

def _O_jvp(forward_fn, params, samples, v):
    _, res = jax.jvp(lambda p: forward_fn(p, samples), (params,), (v,))
    return res


def _O_vjp(forward_fn, params, samples, w):
    _, vjp_fun = jax.vjp(forward_fn, params, samples)
    res, _ = vjp_fun(w)
    return res

def _OH_w(forward_fn, params, samples, w):
    return tree_conj(_O_vjp(forward_fn, params, samples, w.conjugate()))


def _Odagger_DeltaO_v(forward_fn, params, samples, v):
    w = _O_jvp(forward_fn, params, samples, v)
    w = w * (1.0 / (samples.shape[0] * samples.shape[1] * mpi.n_nodes))
    #w_mean = w.sum(axis=(0,1), keepdims=True) / (samples.shape[0] * samples.shape[1] * mpi.n_nodes)
    #w_mean, _ = mpi.mpi_sum_jax(w_mean)
    #w = w - w_mean
    res = _OH_w(forward_fn, params, samples, w)
    return jax.tree_map(lambda x: mpi.mpi_sum_jax(x)[0], res)  # MPI




In [38]:
@partial(jax.jit, static_argnums=0)
def mv(forward_fn, params, samples, v, diag_shift):
    f = lambda p, x: jax.vmap(lambda x: forward_fn({'params': p},x))(x)
    res = _Odagger_DeltaO_v(f, params, samples, v)
    return tree_axpy(diag_shift, v, res)

In [39]:
y1 = jax.block_until_ready(mv(vs1._apply_fun, vs1.parameters, vs1.samples, vs1.parameters, 0.))
y2 = jax.block_until_ready(mv(vs2._apply_fun, vs2.parameters, vs2.samples, vs2.parameters, 0.))

Finished tracing + transforming mv for pjit in 0.03145933151245117 sec
Compiling mv for with global shapes and types [ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(float64[16,512,32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32]), ShapedArray(float64[], weak_type=True)]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(mv) in 0.015076160430908203 sec
Finished XLA compilation of jit(mv) in 0.35547399520874023 sec
Finished tracing + transforming mv for pjit in 0.03065180778503418 sec
Compiling mv for with global shapes and types [ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(float64[8,1024,32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(c

In [40]:
%timeit _ = jax.block_until_ready(mv(vs1._apply_fun, vs1.parameters, vs1.samples, vs1.parameters, 0.))

6.11 ms ± 343 ns per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [41]:
%timeit _ = jax.block_until_ready(mv(vs2._apply_fun, vs2.parameters, vs2.samples, vs2.parameters, 0.))

9.06 ms ± 37.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


- otf is much slower; figure out why

#### pytree

In [42]:
@jax.jit
def mv(S, v):
    return S@v

In [43]:
_  =jax.block_until_ready(mv(S1p, vs1.parameters))
_  =jax.block_until_ready(mv(S2p, vs2.parameters))

Finished tracing + transforming mv for pjit in 0.0006053447723388672 sec
Compiling mv for with global shapes and types [ShapedArray(float64[], weak_type=True), ShapedArray(complex128[16,512,256]), ShapedArray(complex128[16,512,32,256]), ShapedArray(complex128[16,512,32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32])]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated})).
Finished jaxpr to MLIR module conversion jit(mv) in 0.007136344909667969 sec
Finished XLA compilation of jit(mv) in 0.09128618240356445 sec
Finished tracing + transforming fn for pjit in 0.0002903938293457031 sec
Finished tracing + transforming _matmul for pjit in 0.00485992431640625 sec
Finished tracing + transforming mv for pjit in 0.0060880184173583984 sec
Compiling mv for with global shapes and types [Sh

In [44]:
%timeit jax.block_until_ready(mv(S1p, vs1.parameters))

6.66 ms ± 23 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [45]:
%timeit jax.block_until_ready(mv(S2p, vs2.parameters))

4.22 ms ± 8.24 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


- pytree is faster

### vmc

#### pytree

In [46]:
gs1 = nk.VMC(ha, op, variational_state=vs1, preconditioner=srp)
gs2 = nk.VMC(ha, op, variational_state=vs2, preconditioner=srp)

In [47]:
gs1.run(2)
gs2.run(2)

No output specified (out=[apath|nk.logging.JsonLogger(...)]).Running the optimization but not saving the output.


  0%|                                                                                                                                          | 0/2 [00:00<?, ?it/s]Finished tracing + transforming _matmul for pjit in 0.0039520263671875 sec
Finished tracing + transforming real for pjit in 0.000179290771484375 sec
Finished tracing + transforming ravel for pjit in 0.00014662742614746094 sec
Finished tracing + transforming dot for pjit in 0.0003485679626464844 sec
Finished tracing + transforming vdot for pjit in 0.001964092254638672 sec
Finished tracing + transforming imag for pjit in 0.0001666545867919922 sec
Finished tracing + transforming fn for pjit in 0.0002529621124267578 sec
Finished tracing + transforming real for pjit in 0.0002570152282714844 sec
Finished tracing + transforming ravel for pjit in 0.0002048015594482422 sec
Finished tracing + transforming dot for pjit in 0.0003407001495361328 sec
Finished tracing + transforming vdot for pjit in 0.0020935535430908203 sec
Finished trac

No output specified (out=[apath|nk.logging.JsonLogger(...)]).Running the optimization but not saving the output.


  0%|                                                                                                                                          | 0/2 [00:00<?, ?it/s]Finished tracing + transforming _matmul for pjit in 0.0040395259857177734 sec
Finished tracing + transforming _solve for pjit in 0.07598590850830078 sec
Compiling _solve for with global shapes and types [ShapedArray(float64[], weak_type=True), ShapedArray(complex128[8,1024,256]), ShapedArray(complex128[8,1024,32,256]), ShapedArray(complex128[8,1024,32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32]), ShapedArray(complex128[256]), ShapedArray(complex128[32,256]), ShapedArray(complex128[32])]. Argument mapping: (GSPMDSharding({replicated}), GSPMDSharding({devices=[1,2,1]0,1}), GSPMDSharding({devices=[1,2,1,1]0,1}), GSPMDSharding({devices=[1,2,1]0,1}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({replicated}), GSPMDSharding({r

()

In [48]:
gs1.run(100)

No output specified (out=[apath|nk.logging.JsonLogger(...)]).Running the optimization but not saving the output.


100%|█████████████████████████████████████████████████████████████████████| 100/100 [01:56<00:00,  1.17s/it, Energy=-40.7589-0.0008j ± 0.0018 [σ²=0.0241, R̂=1.0405]]


()

In [49]:
gs2.run(100)

No output specified (out=[apath|nk.logging.JsonLogger(...)]).Running the optimization but not saving the output.


100%|█████████████████████████████████████████████████████████████████████| 100/100 [01:02<00:00,  1.60it/s, Energy=-40.7554+0.0006j ± 0.0017 [σ²=0.0220, R̂=1.0674]]


()

In [51]:
1.17 * 1.60 # speedup

1.8719999999999999